# GDGoC UGM Academic Hacker 2026 — Prediksi diagnosis (4 kelas, Macro-F1)

Alur: (1) pemetaan pasien 1:1 → (2) tabel triase long → 1 baris per sampel → (3) gejala: satukan sinonim + flag + keparahan → (4) riwayat medis **hanya sebelum `prediction_time`**: jumlah kejadian + jarak hari ke kejadian terakhir per tabel → (5) HistGradientBoosting + CV OOF → (6) bobot kelas dicari koordinat demi koordinat pada OOF.

Catatan keputusan:
- Teks riwayat (nama penyakit/obat) **tidak dipakai** (TF-IDF sebelumnya menurunkan CV).
- Bobot kelas hanya dipakai bila gain OOF > ~0,001.
- Vitals: koma desimal, unit campur (kPa→mmHg, fraction→%) dan outlier geser desimal dibersihkan.
- `sev_dyspnea` dibuang (tabel severity tidak punya konsep dyspnea); tabel `allergies` hanya 2 baris sehingga dibuang.

In [57]:
import numpy as np
import pandas as pd

DATA_DIR = "."   # <- ubah ke folder hasil ekstrak zip
D = DATA_DIR
TRI = "Triase/Triase/"
HIS = "Riwayat Medis/Riwayat Medis/"

## 1. Pemetaan 1:1 pasien – encounter – sampel

In [58]:
def load_base():
    tr = pd.read_csv("train.csv")
    te = pd.read_csv("test.csv")
    tr["is_train"], te["is_train"] = 1, 0
    base = pd.concat([tr, te], ignore_index=True)

    ie = pd.read_csv("index_encounters.csv")
    pt = pd.read_csv("patients.csv")

    # validasi 1:1 di semua level
    assert base.sample_id.is_unique and base.patient_id.is_unique and base.encounter_id.is_unique
    assert ie.patient_id.is_unique and ie.encounter_id.is_unique and pt.patient_id.is_unique
    assert not set(tr.patient_id) & set(te.patient_id)

    base = base.merge(ie, on=["patient_id", "encounter_id"], how="left", validate="one_to_one")
    assert base.prediction_time.notna().all(), "ada sampel tanpa prediction_time"
    base = base.merge(pt, on="patient_id", how="left", validate="one_to_one")
    base["pred_ts"] = pd.to_datetime(base.prediction_time, utc=True)
    return base


def clean_demo(base):
    df = pd.DataFrame(index=base.index)
    df["age"] = base.age_at_visit
    df["gender"] = base.gender.astype(str).str.strip().str.upper().str[:1].map({"M": 0, "F": 1})

    for c in ["race", "ethnicity"]:
        df[c] = base[c].astype(str).str.strip().str.lower().replace({"nan": "unk"}).astype("category")

    m = base.marital_at_visit.astype(str).str.strip().str.upper().str[:1]
    df["marital"] = m.replace({"N": "unk"}).astype("category")  # 'NAN' -> 'N' -> unk

    def inc(x):
        if pd.isna(x):
            return np.nan
        s = str(x).strip().lower().replace("usd", "").replace("$", "").replace(",", "").strip()
        mult = 1.0
        if s.endswith("k"):
            mult, s = 1000.0, s[:-1]
        try:
            return float(s) * mult
        except ValueError:
            return np.nan

    df["income"] = base.income_at_visit.map(inc)
    df["log_income"] = np.log1p(df.income.clip(lower=0))

    # waktu kunjungan (musiman) - hanya dari prediction_time
    ts = base.pred_ts
    df["pt_month"] = ts.dt.month
    df["pt_dow"] = ts.dt.dayofweek
    df["pt_hour"] = ts.dt.hour
    df["pt_doy"] = ts.dt.dayofyear
    return df

## 2. Tanda vital: long → wide + pembersihan

In [59]:
VITAL_RANGE = {  # rentang fisiologis (setelah konversi ke unit baku)
    "Heart rate": (30, 220), "Respiratory rate": (5, 70), "Systolic BP": (50, 260),
    "Diastolic BP": (20, 160), "SpO2": (50, 100), "BMI": (10, 80),
}


def load_vitals():
    v = pd.read_csv(TRI + "triage_vitals.csv")
    v["x"] = pd.to_numeric(v.value.astype(str).str.replace(",", ".", regex=False), errors="coerce")
    u = v.unit.astype(str).str.lower()

    # geser desimal (x10 / x100) -> bawa ke skala yang benar sebelum konversi unit
    # batas atas per (vital, unit)
    hi = {}
    for name, (lo_, hi_) in VITAL_RANGE.items():
        hi[(name, "std")] = hi_
    hi[("Systolic BP", "kpa")] = 260 / 7.50062
    hi[("Diastolic BP", "kpa")] = 160 / 7.50062
    hi[("SpO2", "fraction")] = 1.0

    def fix(row_name, row_unit, x):
        if np.isnan(x):
            return x
        key = (row_name, row_unit) if (row_name, row_unit) in hi else (row_name, "std")
        for _ in range(3):
            if x > hi[key] * 1.0001:
                x = x / 10.0
            else:
                break
        return x

    v["x"] = [fix(n, uu, x) for n, uu, x in zip(v.vital_name, u, v.x)]
    v.loc[u.eq("kpa"), "x"] = v.loc[u.eq("kpa"), "x"] * 7.50062
    v.loc[u.eq("fraction"), "x"] = v.loc[u.eq("fraction"), "x"] * 100.0
    # buang yang tetap di luar rentang fisiologis
    for name, (lo_, hi_) in VITAL_RANGE.items():
        m = v.vital_name.eq(name) & ((v.x < lo_) | (v.x > hi_))
        v.loc[m, "x"] = np.nan

    # long -> wide, tepat 1 nilai per encounter per vital
    assert not v.duplicated(["encounter_id", "vital_name"]).any()
    w = v.pivot(index="encounter_id", columns="vital_name", values="x")
    w.columns = ["vit_" + c.lower().replace(" ", "_") for c in w.columns]
    return w.reset_index()

## 3. Gejala: sinonim → flag + keparahan

In [60]:
CANON = {
    "throat": ["throat symptoms", "throat pain", "sore throat", "throat discomfort", "painful swallowing"],
    "cough": ["cough", "persistent cough", "cough symptom", "coughing", "cough / productive"],
    "nasal": ["nasal symptoms", "nasal congestion", "nasal complaints", "blocked nose", "stuffy nose"],
    "systemic": ["systemic symptoms", "body malaise", "general malaise", "general fatigue symptoms", "feeling unwell"],
    "fever": ["fever", "raised temperature", "febrile feeling", "feverish", "temperature complaint"],
    "secretions": ["respiratory secretions", "sputum production", "mucus complaint", "airway secretions", "mucus / discharge"],
    "dyspnea": ["dyspnea/wheeze", "difficulty breathing", "breathing discomfort", "wheezy feeling", "shortness of breath / wheeze"],
    "head_face": ["head and facial symptoms"],
}
SYN2CANON = {s: c for c, ss in CANON.items() for s in ss}
CORE = list(CANON)


def canon_symptom(s):
    k = str(s).strip().lower()
    return SYN2CANON.get(k, "other")  # sisa (urinary, thirst, hunger, dst) -> 'other'


def load_symptoms():
    s = pd.read_csv(TRI + "triage_symptoms.csv")
    sv = pd.read_csv(TRI + "triage_symptom_severity.csv")
    s["c"] = s.symptom_concept.map(canon_symptom)
    sv["c"] = sv.symptom_concept.map(canon_symptom)

    flag = s.assign(v=1).pivot_table(index="encounter_id", columns="c", values="v", aggfunc="max", fill_value=0)
    flag = flag.reindex(columns=CORE + ["other"], fill_value=0)
    cnt_other = s[s.c == "other"].groupby("encounter_id").size()
    flag.columns = ["sym_" + c for c in flag.columns]

    sev = sv[sv.c != "other"].pivot_table(index="encounter_id", columns="c", values="severity_score", aggfunc="max")
    sev = sev.reindex(columns=CORE)
    sev.columns = ["sev_" + c for c in sev.columns]

    out = flag.join(sev, how="left")
    out["sym_other"] = cnt_other.reindex(out.index).fillna(0).astype(int)  # jumlah gejala non-inti
    out["sym_n_core"] = out[["sym_" + c for c in CORE]].sum(axis=1)
    out["sev_mean"] = out[["sev_" + c for c in CORE]].mean(axis=1)
    out["sev_max"] = out[["sev_" + c for c in CORE]].max(axis=1)
    out["sev_sum"] = out[["sev_" + c for c in CORE]].sum(axis=1, min_count=1)

    # gejala ada tapi skor keparahan kosong -> NaN dibiarkan (model GBDT menangani)
    return out.reset_index()

## 4. Riwayat per pasien (hanya sebelum prediction_time)

In [61]:
HIST_TABLES = {  # nama -> (file, kolom waktu)
    "prev_enc": ("previous_encounter.csv", "history_start"),
    "cond": ("conditions.csv", "recorded_at"),
    "obs": ("observations.csv", "observed_at"),
    "med": ("medications.csv", "recorded_at"),
    "proc": ("procedures.csv", "recorded_at"),
    "imm": ("immunizations.csv", "recorded_at"),
    "allergy": ("allergies.csv", "recorded_at"),
}


def hist_features(base):
    pred = base.set_index("patient_id").pred_ts
    out = pd.DataFrame({"patient_id": base.patient_id})
    report = {}
    for name, (fn, tcol) in HIST_TABLES.items():
        d = pd.read_csv(HIS + fn)
        d["t"] = pd.to_datetime(d[tcol], utc=True, format="mixed")
        d["pred"] = d.patient_id.map(pred)
        n_all = len(d)
        # kunci anti-leakage: kejadian harus <= prediction_time
        d = d[d.t <= d.pred]
        report[name] = (n_all, n_all - len(d))
        d["gap"] = (d.pred - d.t).dt.total_seconds() / 86400.0
        g = d.groupby("patient_id").agg(n=("gap", "size"), last=("gap", "min"))
        out[f"{name}_n"] = out.patient_id.map(g.n).fillna(0).astype(int)
        out[f"{name}_days_since_last"] = out.patient_id.map(g["last"])  # NaN = tidak pernah ada
    return out.drop(columns="patient_id"), report

## 5. Rakit semua fitur

In [62]:
def build():
    base = load_base()
    demo = clean_demo(base)
    vit = load_vitals()
    sym = load_symptoms()
    hist, rep = hist_features(base)

    X = base[["sample_id", "patient_id", "encounter_id", "is_train", "target_class"]].copy()
    X = pd.concat([X, demo], axis=1)
    X = X.merge(vit, on="encounter_id", how="left", validate="one_to_one")
    X = X.merge(sym, on="encounter_id", how="left", validate="one_to_one")
    hist.index = X.index
    X = pd.concat([X, hist], axis=1)

    # turunan vital
    X["vit_pulse_pressure"] = X.vit_systolic_bp - X.vit_diastolic_bp
    X["vit_map"] = X.vit_diastolic_bp + X.vit_pulse_pressure / 3
    for c in [c for c in X.columns if c.startswith("sym_")]:
        X[c] = X[c].fillna(0)
    assert len(X) == len(base) == 5000
    return X, rep

In [63]:
X, rep = build()
print(X.shape)
print('riwayat dibuang karena > prediction_time (n_total, n_future):', rep)

(5000, 59)
riwayat dibuang karena > prediction_time (n_total, n_future): {'prev_enc': (4471, 0), 'cond': (3583, 0), 'obs': (55648, 0), 'med': (3467, 0), 'proc': (17024, 0), 'imm': (1802, 0), 'allergy': (2, 0)}


## 6. Model, OOF, dan bobot kelas

In [64]:
import numpy as np, pandas as pd, warnings, sys, time
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, log_loss

warnings.filterwarnings("ignore")
CLASSES = ["acute_bronchitis", "strep_pharyngitis", "viral_pharyngitis", "viral_sinusitis"]


def get_xy(X):
    drop = ["sample_id", "patient_id", "encounter_id", "is_train", "target_class", "income",
            "sev_dyspnea",                       # tabel severity tidak punya konsep dyspnea -> kosong
            "allergy_n", "allergy_days_since_last"]  # hanya 2 baris di seluruh data
    feats = [c for c in X.columns if c not in drop]
    return feats


def make_model(seed, **kw):
    p = dict(learning_rate=0.05, max_iter=300, max_leaf_nodes=15, min_samples_leaf=20,
             l2_regularization=1.0, max_features=0.7, early_stopping=False,
             categorical_features="from_dtype", random_state=seed)
    p.update(kw)
    return HistGradientBoostingClassifier(**p)


def oof_probs(Xtr, y, seeds=(0, 1, 2), n_splits=5, **kw):
    P = np.zeros((len(Xtr), 4))
    for s in seeds:
        skf = StratifiedKFold(n_splits, shuffle=True, random_state=s)
        for tr, va in skf.split(Xtr, y):
            m = make_model(s, **kw).fit(Xtr.iloc[tr], y[tr])
            P[va] += m.predict_proba(Xtr.iloc[va])
    return P / len(seeds)


def macro_f1(y, P, w=None):
    w = np.ones(4) if w is None else np.asarray(w)
    return f1_score(y, (P * w).argmax(1), average="macro")


def tune_weights(y, P, grid=None, passes=4):
    """Koordinat demi koordinat: ubah bobot satu kelas, ambil yang memaksimalkan macro-F1 OOF."""
    grid = np.round(np.arange(0.4, 3.01, 0.05), 2) if grid is None else grid
    w = np.ones(4)
    best = macro_f1(y, P, w)
    for _ in range(passes):
        improved = False
        for k in range(4):
            cur = w[k]
            for g in grid:
                w2 = w.copy(); w2[k] = g
                sc = macro_f1(y, P, w2)
                if sc > best + 1e-9:
                    best, w, improved = sc, w2, True
        if not improved:
            break
    return w / w.min(), best

In [65]:
feats = get_xy(X)
tr = X[X.is_train == 1].reset_index(drop=True)
te = X[X.is_train == 0].reset_index(drop=True)
y = tr.target_class.map({c: i for i, c in enumerate(CLASSES)}).values
CFG = dict(learning_rate=0.03, max_iter=250, max_leaf_nodes=8, min_samples_leaf=30, l2_regularization=3.0)

P = oof_probs(tr[feats], y, seeds=(0, 1, 2), **CFG)
print("OOF macro-F1 (argmax):", round(macro_f1(y, P), 4), "| logloss:", round(log_loss(y, P), 4))

w, best = tune_weights(y, P)
gain = best - macro_f1(y, P)
print("bobot:", dict(zip(CLASSES, w.round(2))), "| OOF F1:", round(best, 4), "| gain:", round(gain, 4))
use_w = w if gain > 0.0015 else np.ones(4)   # gain ~0,001 -> diabaikan

OOF macro-F1 (argmax): 0.6775 | logloss: 0.4474
bobot: {'acute_bronchitis': np.float64(1.0), 'strep_pharyngitis': np.float64(3.75), 'viral_pharyngitis': np.float64(1.67), 'viral_sinusitis': np.float64(1.67)} | OOF F1: 0.7156 | gain: 0.0381


In [67]:
from catboost import CatBoostClassifier

Pt = np.zeros((len(te), 4))
for s in range(10):
    Pt += CatBoostClassifier(s, **CFG).fit(tr[feats], y).predict_proba(te[feats])
Pt /= 10
pred = (Pt * use_w).argmax(1)
sub = pd.DataFrame({"sample_id": te.sample_id, "target_class": [CLASSES[i] for i in pred]})
sub.to_csv("submission_2_v1.csv", index=False)
print(sub.target_class.value_counts(normalize=True).round(3))

TypeError: CatBoostClassifier.__init__() got an unexpected keyword argument 'max_iter'